# Lab 2 Agent Pipeline (experiments and retrieval over your saved pipeline)
**Objective:** direct an agent through classification experiments and retrieval comparisons over the pipeline you built and saved in `DM2026-Lab2-Master.ipynb`, and audit what it reports.

This notebook comes after the Master. There you built five representations of the same tweets, trained classifiers on them, and measured retrieval; the Master's Section 9 saved all of it to `lab2_workspace/`. Here an agent runs experiments on that saved pipeline under your direction. It is the same kind of agent as in the Lab 1 agentic notebooks, built on the same `agent_pipeline/` package: one tool call per message, a mandatory plain-text comment after every tool call, an approval step before any tool that computes something new, and a check on every number it states.

Nothing here re-teaches the Master. The agent runs and compares; your part is to decide what is worth running, one controlled step at a time, and to check every claim against the tool output that produced it.

## Table of Contents
0. Setup: your name, a backend, and its API key
1. Load the saved pipeline
2. What the agent can do, and what to check
3. Build the graph and launch the chat
4. Exercises (14 to 16)
5. Submitting your session log

## 0. Setup: your name, a backend, and its API key
**Objective:** identify yourself for the session log and choose the language model that powers the agent.

Your name and student ID come from `config/.env` — the same git-ignored file as your API key — so nothing identifying is typed into this notebook or saved in its outputs. Fill in `STUDENT_NAME` and `STUDENT_ID` there if you have not already. Both are encrypted into your session log, which is how it gets matched to you however the file is renamed. The backends and the `config/.env` key are the same as in Lab 1 (see `config/.env.example`). **Set up both providers if you can.** Either one answers a tool-calling turn in a few seconds, but a single provider runs out of its per-minute token budget quickly here: every message the agent handles resends the system prompt and all fourteen tool descriptions, so one key alone stalls after a couple of messages. With both configured and fallback on, seven messages in a row go through with no waiting, the chat switches providers when one budget ran out and switches back a minute later once it recovered.

**A second provider helps more than a second key of the same kind.** Groq applies its rate limit to your whole account, so extra Groq keys add no quota. Extra Gemini keys help only when each comes from a separate Google project, since each project has its own limit. One key from each provider is the combination that keeps this notebook moving.

If you have keys for both providers, **enable multi-provider fallback**. When a backend hits a rate limit, or its servers are unreachable or overloaded, the agent retries on another key or provider instead of stopping, and the chat prints a line whenever it switches. The limit you will actually meet is a per-minute one, and it clears within about a minute. The daily quotas are much larger and reset once a day (Groq at midnight UTC, Gemini at midnight Pacific time). It's recommended to start with time to spare, just in case.

You can type `/status` in the chat at any point to see which of your keys are still active, which have hit a rate limit this session, and how large the requests are getting. The chat widget answers that itself: it never reaches the agent, costs no turn, and is not written to your session log. `/keys` does the same thing.

Run the cell below, fill it in, and click **Confirm** before moving on.

In [1]:
from agent_pipeline.backend_widget import select_backend

backend_choice = select_backend()

## 1. Load the saved pipeline
**Objective:** load the agent's tools and the saved pipeline from disk, so this notebook starts from a fresh kernel and still works on exactly what the Master built.

The tools come from two folders. `lab2_tools/inspection/` holds tools that only read what is already saved (the list of representations, the Master's results table, breakdowns of earlier results). They compute nothing new and send no tweets to the model, so they are usable immediately. `lab2_tools/experiments/` holds tools that train a classifier, run retrieval, or show example tweets; those need your explicit approval before they run, through the same request and approval exchange as Lab 1's agentic pipeline, and one approval covers all five of them at once rather than one tool at a time.

Every tool reads `lab2_workspace/`, never this kernel's memory. That is why the agent's numbers can be checked against the Master: a run with the Master's settings reproduces the Master's score to within a prediction or two. If the cell below reports that the workspace is missing, run `DM2026-Lab2-Master.ipynb` through Section 9 first.

In [2]:
import os

from agent_pipeline.session_state import SessionState
from agent_pipeline.workspace_loader import load_workspace_tools
from agent_pipeline.tool_access import build_gated_tools

if not os.path.isfile("lab2_workspace/manifest.json"):
    print("lab2_workspace/ not found. Run DM2026-Lab2-Master.ipynb through Section 9 first, then re-run this cell.")

session = SessionState()
experiment_tools, experiment_problems = load_workspace_tools("lab2_tools/experiments", session)
inspection_tools, inspection_problems = load_workspace_tools("lab2_tools/inspection", session)
tools = build_gated_tools(experiment_tools, session, premade_tools=inspection_tools)
problems = experiment_problems + inspection_problems

print(f"Experiment tools ({len(experiment_tools)}, need your approval to run): {[t.name for t in experiment_tools]}")
print(f"Inspection tools ({len(inspection_tools)}, usable right away): {[t.name for t in inspection_tools]}")
if problems:
    print("\nNot loaded:")
    for fname, message in problems:
        print(f"  {fname}: {message}")

Experiment tools (5, need your approval to run): ['run_classifier_tool', 'show_misclassified_tool', 'retrieve_tool', 'compare_retrieval_tool', 'evaluate_retrieval_tool']
Inspection tools (4, usable right away): ['list_representations_tool', 'show_master_results_tool', 'inspect_result_tool', 'compare_results_tool']


## 2. What the agent can do, and what to check
**Objective:** know the tools well enough to direct them precisely, and know what to verify in the agent's replies before trusting them.

| Tool | Needs approval | What it does |
|---|---|---|
| `list_representations_tool` | no | Lists the five saved representations and the dataset's schema (column names and types, row and label counts). |
| `show_master_results_tool` | no | The Master's classification results table, best first: the baseline to compare against. |
| `inspect_result_tool` | no | Per-emotion precision, recall, and F1, the confusion matrix, and the most frequent confusion for one classifier run. |
| `compare_results_tool` | no | Two or more classifier runs side by side, with the gap between best and worst. |
| `run_classifier_tool` | yes | Trains KNN, a linear SVM, Naive Bayes, or an MLP on one representation and evaluates it on the test tweets. |
| `show_misclassified_tool` | yes | Up to 10 test tweets one run got wrong, with true and predicted emotion and the annotated intensity. |
| `retrieve_tool` | yes | The top *k* training tweets for a query under one retriever (TF-IDF, Word2Vec, or MiniLM). |
| `compare_retrieval_tool` | yes | One query through all three retrievers, with how many documents each pair shares. |
| `evaluate_retrieval_tool` | yes | Label precision@*k* for one retriever over all test queries, with the random baseline. |

The agent also has the five access tools you know from Lab 1 (list, describe, request, grant, discard).

**The first experiment tool you ask for takes three messages of your own, and it does not run until the third.** The graph allows the agent one tool call per message, and granting access is itself a tool call, so approving and running cannot happen together. The approval is a batch: it covers all five experiment tools at once, so this exchange happens once per session, not once per tool:

| Your message | What the agent does | What you should see |
|---|---|---|
| 1. Ask for the step, with the details specified | Calls `request_tool_access_tool` and explains why | A pending request. No result yet. |
| 2. Approve it (for example "yes, go ahead") | Calls `grant_tool_access_tool` | A confirmation that all five experiment tools are now usable. **Still no result:** nothing has run yet. |
| 3. Ask for the run again | Calls the tool itself | The result, with a `result_id` |

After that, every experiment tool runs directly for the rest of the session, one call per message, including the ones you have not asked for yet. The agent will not request and grant in one step, and it will not chain two steps into one message, however the request is phrased.

**If approving seems to do nothing.** Approving produces a confirmation, not a result, so message 2 always looks quiet. When the agent answers with nothing but the tool call it wants next, the chat prints that step instead of a blank reply, as `(Done with that step. Next it wants to run ... Say "go ahead" to run it, or ask for something else.)`, which is your cue to send the next message. If the agent replies with a comment but never calls `grant_tool_access_tool`, repeat the approval. If it tries to run the tool in the same message as the grant, the run is refused (it was not granted yet) and the grant is dropped, so nothing changes at all: approve on its own, and ask for the run in a separate message.

**What to check in every reply.** The agent's prose is a claim about tool output, not the output itself. The tables in the chat come directly from the tool; the sentences around them come from the model. Each time, check:
- that every number it states appears in the tool output above it (the chat marks any number it cannot trace, and the mark is also recorded in your log);
- that it describes the tweets actually shown, not tweets it imagines (compare its summary with the table);
- that a claimed improvement is larger than the differences noise alone would produce; on 347 test tweets, a gap in the third decimal is not a finding;
- that a run with the Master's settings reports a `master_reference_macro_f1` within a prediction or two of its own `macro_f1`, and that you find out why if the gap is wider.

**What your log records.** Each message you send is logged with the tool call it triggered (name and arguments), the tool's result, and the agent's reply, encrypted as it is written. Grading reads that sequence: specific, one-step requests that build on earlier results score well; vague requests, requests copied from this notebook, and approvals given without reading what was proposed do not.

**If the agent's reply is only "(No response came back for that...)".** The tool call already ran and its result is shown above; only the model's comment on it came back empty. Ask it to summarize the result, or continue with your next request. Nothing needs restarting.

**If replies seem to be missing entirely, check your widget versions.** With `ipywidgets` 8.x the chat's input row renders inside the scrollable conversation box, so new replies can appear underneath it and look absent. From the repository root, run `uv sync --locked --python 3.11` to restore the pinned `ipywidgets==7.8.1` environment, then restart Jupyter with `uv run --locked jupyter lab`.

**If the chat does not appear at all, restart JupyterLab itself, not just the kernel.** Now and then the launch cell renders an error or nothing where the chat should be, and restarting the kernel does not clear it, because the stale widget state lives in the browser tab and the running server rather than in the kernel. Close the JupyterLab tab, stop the server in the terminal you started it from (`Ctrl+C`), start it again with `uv run --locked jupyter lab`, and run the notebook from the top. The chat renders normally after that, and nothing you have already logged is lost.

**If the chat reports a rate limit or an unreachable provider, or tells you to wait and send another message.** Every configured key and provider failed in a row, and the chat handed control back to you rather than freezing the notebook. Do not restart or stop the kernel: the retry progress lives only in its memory, and a restart starts the count over. Wait a few minutes and send your next message; the retries continue where they stopped. Type `/status` to see which keys are still active before you spend a turn on a real request.


## 3. Build the graph and launch the chat
**Objective:** connect the tools, the rules, and your chosen model into the agent, and open the chat.

The graph is the one Lab 1's agentic pipeline uses (`agent_pipeline/graph_core.py`): one tool call per message, then a forced plain-text comment on the result, then a stop to wait for you. The system prompt at the top of the cell states the rules the agent works under; read it once, since it defines what you can reasonably ask for. Re-running this cell starts a fresh conversation, and the log keeps the earlier one.

In [3]:
SYSTEM_PROMPT = """You run experiments over the student's saved Lab 2 pipeline. \
DM2026-Lab2-Master.ipynb saved it to lab2_workspace/: five representations \
(tfidf, word2vec, doc2vec, fasttext, minilm) of the same SemEval-2017 emotion \
tweets (anger, fear, joy, sadness), the fitted models, and the Master's own \
results table. Your tools read those files. You never see the dataset itself, \
only small summaries and, when the student asks, at most ten example tweets.

Inspection tools (list_representations_tool, show_master_results_tool, \
inspect_result_tool, compare_results_tool) only read saved results and are \
usable immediately. Experiment tools (run_classifier_tool, \
show_misclassified_tool, retrieve_tool, compare_retrieval_tool, \
evaluate_retrieval_tool) compute new results or send tweets to you, and need \
the student's approval first, granted once as a batch covering all five. \
Calling one before that returns a refusal, not a result. Where the access \
tools say "the student's own tools", they mean these five.

Rules you must follow:
1. Call AT MOST ONE tool per response, and only when the student has asked for \
a specific action with the details it needs.
2. If a request is ambiguous (which representation, classifier, setting or \
query), ask instead of guessing.
3. While the batch is ungranted, call request_tool_access_tool with a reason \
tied to what the student asked, then stop and wait. It takes only that reason \
and proposes all five at once. When the student's next message approves, call \
grant_tool_access_tool as that turn's ONLY tool call: never in the same turn \
as the request, and never in the same turn as the tool being granted, since \
only your first tool call in a turn runs. Never answer an approval with plain \
text alone; make the grant call. That one grant makes all five usable, so say \
so and name the exact request to send next. Once granted, never call \
request_tool_access_tool again. If the student declines, call \
discard_tool_request_tool.
4. Answer what tools exist or what one does with list_available_tools_tool or \
describe_tool_tool, never from a name alone.
5. After a tool returns, report what it shows, quoting the actual values and \
doc_ids rather than paraphrasing them.
6. Every number you state must come from a tool's returned data. Do not \
compute, estimate or invent one, and do not call a difference an improvement \
without stating its size.
7. Refuse to chain several steps in one message; the student directs each step.
8. Refer to an earlier result by its exact result_id.
9. Right after a tool result you get exactly one turn to comment in plain \
text, with no tool call of any kind. That applies only to that turn; the \
student's next message allows a tool call again.
"""

from agent_pipeline.graph_core import build_agent_graph
from agent_pipeline.chat_widget import launch_chat

if "backend" not in backend_choice:
    raise RuntimeError("Pick a backend in the setup cell above and click Confirm before running this one.")
LLM_BACKEND = backend_choice["backend"]
MULTI_PROVIDER_MODE = backend_choice["multi_provider"]
STUDENT_NAME = backend_choice["student_name"]
STUDENT_ID = backend_choice["student_id"]

if MULTI_PROVIDER_MODE:
    from agent_pipeline.llm_fallback import build_fallback_llm
    llm, active_chain = build_fallback_llm(LLM_BACKEND, temperature=0)
    print(f"Fallback chain (in order): {active_chain}")
else:
    from agent_pipeline.llm_backends import build_llm_for_backend
    llm = build_llm_for_backend(LLM_BACKEND, temperature=0)

app = build_agent_graph(tools, SYSTEM_PROMPT, llm, session)

graph_state, logger = launch_chat(
    app, session, log_path="session_logs/lab2_agent_pipeline_session.jsonl.enc",
    student_name=STUDENT_NAME, student_id=STUDENT_ID, track="lab2-agent-pipeline",
    placeholder="Ask the agent to run one experiment... (or type /status to see key rotation status)",
    llm=llm,
)

## 4. Exercises
**Objective:** practice directing controlled experiments through the agent and auditing its reports against tool output.

Answer each exercise in the cell below it, citing the `result_id`s from your conversation. The graded evidence is what you write here together with the logged conversation that produced it.

### **>>> Exercise 14 (take home):**
**Purpose:** beat a number the Master already obtained, and claim no more than the margin allows.

Pick any row of the Master's results table and try to beat its macro F1. Reproduce it first (same representation, classifier and settings), and check your `macro_f1` against the `master_reference_macro_f1` beside it. That is your baseline. Then make your attempts, one run per message. Ask the agent what `run_classifier_tool` can vary, and count the table: seventeen rows, but five representations times four classifiers is twenty. Four or five runs is enough; finish with `compare_results_tool`.

In the cell below, give your baseline and best attempt with their `result_id`s and say whether you beat the Master. The Master read 0.006 macro F1 as three of the 347 test predictions (Section 7); convert your margin the same way and say whether it is a finding or a coin flip. **If nothing beat the Master's setting, say so and show the runs**. Close with two sentences: one quoting an agent claim you checked yourself, and one on whether your best score is a fair estimate for new tweets, given that you chose it on the test set.

*What to check in the agent's output:* each run changed only what you asked, and its summary of the `compare_results_tool` table matches the table.

**Summary:** My baseline is `classify_3` (TF-IDF + Linear SVM, C = 1.0, macro F1 = 0.8621), my best alternative attempt is `classify_7` (C = 0.75, macro F1 = 0.8596), and I did not beat the Master.

### 1. Baseline Selection and Reproducibility

I selected TF-IDF with Linear SVM as my baseline because it achieved the highest macro F1 of 0.8621 in the Master notebook. In Exercise 9, adding 8,505 bigram features reduced the macro F1 from approximately 0.862 to 0.831, suggesting that increasing feature complexity might introduce noise or overfitting. Therefore, I decided to investigate regularization strength instead of expanding the feature space.

After inspecting `master_results_2`, I confirmed that the Master results table contains 17 rows, although five representations combined with four classifiers give 20 possible combinations. The three missing combinations are Word2Vec + MLP, Doc2Vec + MLP, and FastText + MLP, because the Master notebook only evaluated MLP with TF-IDF and MiniLM. The agent's tool supports these unexplored combinations, although their `master_reference_macro_f1` would be null. By inspecting the description of `run_classifier_tool`, I also confirmed that the adjustable hyperparameters are `k` for KNN, `C` for Linear SVM, and `hidden_layer_sizes` for MLP.

I first reproduced the Master baseline using TF-IDF, Linear SVM, and C = 1.0 (`classify_3`). The reproduced accuracy was 0.8617, while both `macro_f1` and `master_reference_macro_f1` were 0.8621. The difference was zero, confirming successful reproduction of the Master's reported baseline performance.

### 2. Controlled Experiments and Results

My initial hypothesis was that stronger regularization might improve generalization by reducing the model's dependence on rare and sparse features. However, reducing C to 0.5 decreased the macro F1 to 0.8503. This result did not support my hypothesis that the unigram-based baseline would benefit from stronger regularization. An alternative hypothesis is that the performance decline in Exercise 9 resulted from L2 normalization distributing feature weights across many additional low-frequency bigrams. However, this explanation was not directly tested in my current experiments.

Based on the first result, I explored the opposite direction by increasing C to 2.0, which also produced a lower macro F1 than the baseline. I then tested C = 1.5 and C = 0.75 to investigate the region closer to the original setting. Throughout these experiments, I kept the representation, classifier, and other model settings unchanged. I checked the returned `representation`, `classifier`, and `parameters` fields of every run to confirm that only the requested C value changed.

The final results obtained using `compare_results_tool`(`compare_8`) were:

| Result ID | C | Accuracy | Macro F1 |
|---|---:|---:|---:|
| classify_3 | 1.00 | 0.8617 | **0.8621** |
| classify_7 | 0.75 | 0.8588 | 0.8596 |
| classify_5 | 2.00 | 0.8530 | 0.8530 |
| classify_4 | 0.50 | 0.8501 | 0.8503 |
| classify_6 | 1.50 | 0.8501 | 0.8503 |

None of my alternative configurations outperformed the Master baseline. The best overall result remained `classify_3` with a macro F1 of 0.8621, while the best alternative was `classify_7` with a macro F1 of 0.8596.

I verified that the agent's reported ranking, `best_result_id`, `macro_f1_gap`, and tied worst-performing configurations agreed with the comparison table. However, the agent also claimed that tied results were sorted by accuracy. This claim could not be verified from the table because `classify_4` and `classify_6` had identical accuracy and macro F1. Furthermore, the implementation in `inspection_tools.py` sorts results only by macro F1, without using accuracy as a secondary sorting criterion.

Although C = 1.0 achieved the highest overall macro F1, it did not perform best for every emotion. For example, C = 2.0 achieved the highest anger F1 of 0.8402, while C = 0.75 achieved the highest sadness F1 of 0.8456. Moreover, `classify_4` and `classify_6` achieved identical accuracy and macro F1 despite having different per-class F1 scores. These observations demonstrate that aggregate metrics can hide differences in class-level performance.

### 3. Performance Margin: A Finding or a Coin Flip?

There was no improvement to convert: the best alternative (`classify_7`) fell 0.0025 short of the baseline (`classify_3`), while the gap between the best and worst configurations was 0.0118.

Following the approximate interpretation in Section 7 of the Master notebook, where a macro F1 difference of 0.006 corresponded to three predictions, the observed differences can be interpreted as follows:

- Best alternative versus baseline: (0.0025 / 0.006) × 3 = 1.25, approximately one prediction.
- Best versus worst configuration: (0.0118 / 0.006) × 3 = 5.9, approximately six predictions.

However, converting the reported accuracy scores into correct prediction counts provides a different perspective:

| Result ID | C | Correct Predictions (out of 347) | Difference from Baseline |
|---|---:|---:|---:|
| classify_3 | 1.00 | 299 | 0 |
| classify_7 | 0.75 | 298 | −1 |
| classify_5 | 2.00 | 296 | −3 |
| classify_4 | 0.50 | 295 | −4 |
| classify_6 | 1.50 | 295 | −4 |

The macro F1 gap of 0.0118 would correspond to approximately six predictions using the Master's conversion, yet the accuracy counts differ by only four correct predictions. This illustrates that macro F1 cannot be directly converted into an exact number of correct predictions because it depends on class-level precision and recall rather than simply counting correct classifications. Furthermore, a net difference in correct predictions can hide a larger number of individual predictions that changed, since newly correct and newly incorrect predictions may offset each other.

The relationship between C and macro F1 was also not monotonic. The score decreased from 0.8621 at C = 1.0 to 0.8503 at C = 1.5, but subsequently increased to 0.8530 at C = 2.0. Although different C values produced different observed test scores, these results do not establish a consistent relationship or a universally optimal C value.

Given that the observed differences correspond to a net difference of only one to four correct predictions out of 347 test tweets, I would interpret these results as closer to a coin flip than a meaningful finding. Without additional validation or statistical testing, there is insufficient evidence to distinguish a genuine generalization advantage from sampling variability.

### 4. Agent Claim Verification

One agent claim I personally checked was “Joy F1 fell from 0.9161 (classify_3) to 0.8974 (classify_4),” which was incorrect because the actual joy F1 of `classify_4` was 0.8831, while 0.8974 belonged to `classify_5` and `classify_6`, demonstrating that even a valid numerical value can be incorrectly attributed to another experiment.

### 5. Test-Set Selection Bias

Although C = 1.0 was the Master's preset default rather than a hyperparameter I selected after observing my new experimental results, I do not consider the best macro F1 of 0.8621 an unbiased estimate of performance on unseen tweets because TF-IDF with Linear SVM was originally selected from 17 models evaluated on the same test set and my subsequent hyperparameter comparisons reused that test set, potentially introducing selection bias; a more reliable approach would be to select models and hyperparameters through cross-validation on the training set and evaluate the final selected model only once on an independent test set.

### **>>> Exercise 15 (take home):**
**Purpose:** judge what three representations retrieve for the same query, and weigh one query's verdict against an average over 347.

1. Write two queries of your own about a topic this dataset contains. Predict which of the three retrievers will serve each better, and report where you were wrong.
2. Have the agent run `compare_retrieval_tool` on each, one per message. Read the three tables of retrieved tweets yourself before reading the agent's summary.
3. Have it run `evaluate_retrieval_tool` for TF-IDF and MiniLM at the same *k*. It repeats the Master's Section 8.2 computation, so check its numbers against the table you printed there.

In the cell below, for each query: which retriever's list would you use, citing `doc_id`s, and does the agent's summary of it match the table? What does Word2Vec's list say about that representation? Your queries may make one retriever look far better than another while precision@*k* separates them by a few hundredths: explain how both can be true.

*What to check in the agent's output:* the overlap counts it quotes match the `overlap` table; it does not present one query's result as a general conclusion.

**Summary:** I would choose MiniLM for both queries, although for different reasons. For Query 1, MiniLM provided greater retrieval diversity despite receiving a slightly lower manual relevance score than TF-IDF. For Query 2, MiniLM was the only retriever that returned topically relevant documents. However, across 347 test queries, MiniLM's overall label precision@5 exceeded TF-IDF's by only 0.0254, illustrating the difference between individual retrieval usefulness and aggregate label-based evaluation.

### 1. Query Design, Predictions, and Evaluation Criteria

I designed two queries about job interview anxiety, keeping the intended topic and target emotion (fear) consistent while changing how they were expressed.

**Query 1 — Lexical matching**

"I feel nervous and have anxiety before a job interview."

**Prediction: TF-IDF > MiniLM > Word2Vec.**

I expected TF-IDF to perform best because explicit keywords such as "nervous," "anxiety," "job," and "interview" favor lexical matching.

**Query 2 — Semantic matching**

"My hands are shaking as I wait to be called in by the person who will decide whether to hire me."

**Prediction: MiniLM > TF-IDF > Word2Vec.**

I expected MiniLM to recognize the underlying interview scenario, while TF-IDF might retrieve unrelated tweets sharing individual words, and Word2Vec might lose contextual information through word-vector averaging.

I initially considered exam anxiety, but a preliminary vocabulary inspection of the saved TF-IDF vectorizer and Word2Vec model (performed outside the agent) indicated that "anxious" and "exam" were out of vocabulary, so I changed the topic.

Before conducting the experiments, I established the following manual relevance criteria:

| Score | Relevance | Definition |
|---:|---|---|
| 2 | Relevant | Directly relates to job interviews, regardless of the emotion label. |
| 1 | Partially relevant | Expresses fear, nervousness, or anxiety in the text but is unrelated to job interviews. |
| 0 | Irrelevant | Unrelated to both the intended topic and emotion. |

This supplementary metric evaluates tweet content rather than annotated labels, with a maximum score of 10 per retrieved list. One limitation is that Query 2 is longer than Query 1, introducing a confounding variable beyond paraphrasing.

### 2. Query 1: Lexical Retrieval

**Result ID:** `compare_retrieval_9`

| Metric | TF-IDF | Word2Vec | MiniLM |
|---|---:|---:|---:|
| Manual relevance score (/10) | 10 | 2 | 9 |
| Topically relevant documents | 5 | 0 | 4 |
| Fear-labeled documents | 5 | 2 | 5 |
| Substantially distinct contents | 3 | 5 | 5 |

**Retriever choice and prediction evaluation.** I would choose MiniLM despite TF-IDF's slightly higher manual relevance score. TF-IDF retrieved two near-duplicate pairs (`20158` and `20865`, `20014` and `20140`), differing mainly in the presence of #nervous. Its strong lexical matching therefore produced only three substantially distinct pieces of content.

MiniLM additionally retrieved `20195`, which describes nervousness and excitement while waiting for an interview outcome, and `20106`, which mentions an unspecified interview. I assigned `20106` one point as a conservative judgment under uncertainty, since my criteria did not define a category for an interview of unspecified type. Assigning it two points would give MiniLM a perfect score.

Although my predicted score ranking was correct, I underestimated MiniLM's practical usefulness. The evidence only partially supports TF-IDF's predicted advantage because its higher score depends on a subjective judgment and does not account for redundancy.

Word2Vec retrieved general anxiety (`20011`) and procrastination-related panic (`20010`), but neither concerns job interviews.

**Agent verification.** I confirmed that the reported emotion counts and overlap counts matched the tool output.

| Retriever pair | Shared documents | Shared doc_ids |
|---|---:|---|
| TF-IDF vs. Word2Vec | 0 | None |
| TF-IDF vs. MiniLM | 3 | 20014, 20104, 20158 |
| Word2Vec vs. MiniLM | 0 | None |

However, the agent's claim that all TF-IDF and MiniLM results mentioned job interviews was too broad because `20106` does not specify the interview type. It also recognized the near-duplicates without fully discussing their impact on retrieval usefulness.

During my final verification, the agent claimed that the earlier comparison results were "not stored." However, the implementation of `compare_retrieval_tool` explicitly calls `session.store_result()`. The agent may have lacked access to earlier results within its active context, but lack of access is not equivalent to lack of storage.

### 3. Query 2: Semantic Retrieval

**Result ID:** `compare_retrieval_10`

| Metric | TF-IDF | Word2Vec | MiniLM |
|---|---:|---:|---:|
| Manual relevance score (/10) | 2 | 1 | 10 |
| Topically relevant documents | 0 | 0 | 5 |
| Fear-labeled documents | 2 | 2 | 5 |
| Substantially distinct contents | 5 | 5 | 3 |

**Retriever choice and prediction evaluation.** I would choose MiniLM because it was the only retriever that returned job-interview-related tweets. Documents `20014` and `20158` describe upcoming interviews, while `20195` describes waiting for an interview outcome. These three documents also appeared in MiniLM's Query 1 results, suggesting consistency across different query wordings.

However, `20014` and `20140`, as well as `20195` and `20971`, are near-duplicate pairs. MiniLM therefore returned only three substantially distinct pieces of content.

TF-IDF demonstrated lexical overlap without topical relevance:

| Doc ID | Main shared words with Query 2 | Actual topic |
|---|---|---|
| 10419 | called, whether | An offensive remark on a train |
| 20100 | hands, who | Violence and personal safety |
| 40132 | whether, to | Despair and uncertainty |
| 10657 | wait, to, be, in, my | A Halloween costume |
| 20101 | hands, shaking | Emotional distress |

Word2Vec also retrieved no interview-related tweets. Although `20080` and `21035` are both labeled fear, only `20080` explicitly describes nervousness.

My predicted ranking was correct, although the one-point difference between TF-IDF and Word2Vec is too dependent on subjective interpretation to support a strong conclusion. Moreover, although both returned more diverse lists than MiniLM, their results were off-topic, demonstrating that greater diversity does not necessarily imply higher retrieval quality.

**Agent verification.** All three cross-retriever overlap counts were zero, and the reported emotion counts and near-duplicate pairs were correct. However, I identified four issues:

- The agent claimed that all MiniLM results contained anxiety cues, although `20971` and `20140` do not explicitly express nervousness.
- It treated `21035` as emotionally relevant based on its fear label rather than its content.
- It described `20101` as expressing "extreme fear," although shaking hands could indicate other forms of emotional distress.
- Its reported 40% fear proportions were mathematically correct but were not directly returned by the tool, violating Rule 6.

Although the results support my semantic matching hypothesis, MiniLM may have relied substantially on the word "hire." Removing this word in an additional ablation experiment could test this alternative explanation, but I did not conduct that experiment.

### 4. What Word2Vec's Results Reveal

Across both queries, Word2Vec retrieved ten documents, none of which was topically relevant, and only three received one point in my manual assessment.

| Retriever | Q1 Rank 1 Similarity | Q2 Rank 1 Similarity |
|---|---:|---:|
| TF-IDF | 0.5079 | 0.2405 |
| Word2Vec | 0.9670 | 0.9629 |
| MiniLM | 0.6302 | 0.5051 |

I compare similarity changes within each representation rather than absolute scores across different vector spaces.

Word2Vec's scores remained concentrated between approximately 0.946 and 0.967 across both queries. For example, `30255`, a joy-labeled tweet about playful social interaction, received a similarity of 0.9498 despite being unrelated to interview anxiety.

This suggests limited discriminative ability in the tested retrievals, consistent with Master Section 8.2, where Word2Vec's P@5 was 0.283, only slightly above the random baseline of 0.256.

Possible explanations include the loss of word order and contextual information through vector averaging, combined with the small training corpus of 3,606 tweets. The external vocabulary inspection indicated that Query 1 had no OOV words and Query 2 had only one ("hire"), making OOV an unlikely primary explanation.

These explanations remain hypotheses because their individual effects were not experimentally isolated.

### 5. Aggregate Retrieval Evaluation

I evaluated TF-IDF and MiniLM using `evaluate_retrieval_tool` under identical settings.

| Metric | TF-IDF | MiniLM |
|---|---:|---:|
| Result ID | evaluate_retrieval_11 | evaluate_retrieval_12 |
| Number of queries | 347 | 347 |
| k | 5 | 5 |
| Master P@5 | 0.523 | 0.549 |
| Measured P@5 | 0.5233 | 0.5487 |
| Random baseline | 0.2563 | 0.2563 |

| Emotion | TF-IDF P@5 | MiniLM P@5 | Difference (MiniLM − TF-IDF) |
|---|---:|---:|---:|
| Anger | **0.5667** | 0.5571 | −0.0096 |
| Fear | 0.5055 | **0.5836** | +0.0781 |
| Joy | 0.5013 | **0.5342** | +0.0329 |
| Sadness | **0.5243** | 0.5027 | −0.0216 |
| Overall | 0.5233 | **0.5487** | +0.0254 |

Both evaluations reproduced Master Section 8.2 within rounding differences, including the random baseline of 0.256. MiniLM achieved an overall advantage of 2.54 percentage points and performed better for fear and joy, while TF-IDF performed better for anger and sadness; however, no statistical significance test was conducted.

### 6. Why Individual and Aggregate Retrieval Performance Can Differ

MiniLM achieved 10/10 in Query 2, compared with TF-IDF's 2/10, yet their aggregate label precision@5 differed by only 0.0254. These observations can coexist for four reasons.

**Reason 1 — Different definitions of relevance**

Label precision@k measures emotion label agreement, whereas my manual assessment prioritizes topical relevance. Using fear as the intended target emotion for Query 2 (`compare_retrieval_10`) illustrates the difference:

| Metric | TF-IDF | MiniLM | Difference |
|---|---:|---:|---:|
| Topical relevance | 0/5 = 0.0 | 5/5 = 1.0 | 1.0 |
| Target emotion label agreement | 2/5 = 0.4 | 5/5 = 1.0 | 0.6 |
| Manual relevance score | 2/10 | 10/10 | 8 points |

Documents `20100` and `20101` receive credit under fear-label agreement despite being unrelated to job interviews, as would `20011` and `20010` from `compare_retrieval_9`. Label precision also counts near-duplicates separately, regardless of whether they provide additional information.

**Reason 2 — Different query characteristics**

The aggregate evaluation uses actual test tweets, whereas Query 2 was deliberately paraphrased to avoid explicit topic and emotion keywords. It is possible that test tweets containing explicit emotion words or hashtags favor TF-IDF, but neither my experiments nor the available tool outputs establish how common such expressions are. The agent correctly identified this explanation as an unverified hypothesis.

**Reason 3 — Variation across individual queries**

In `compare_retrieval_9`, TF-IDF and MiniLM both retrieved five fear-labeled documents, with three shared documents (`20014`, `20104`, `20158`) and similar manual relevance scores. The substantial topical relevance gap occurred only in Query 2 (`compare_retrieval_10`) among my two custom queries, neither of which was part of the aggregate test evaluation.

**Reason 4 — Aggregation across queries and emotions**

The overall difference corresponds to approximately:

0.0254 × 347 × 5 ≈ 44

This represents a net advantage of approximately 44 label-matching retrieval positions out of 1,735, not 44 additional unique relevant tweets.

The per-emotion results from `evaluate_retrieval_11` and `evaluate_retrieval_12` demonstrate that MiniLM's advantages for fear and joy are partially offset by TF-IDF's advantages for anger and sadness. However, the tool reports only overall and per-emotion precision, so dilution across individual queries remains a plausible but untested explanation.

**Final agent verification**

The agent incorrectly connected MiniLM's stronger fear precision to the interview-scenario query "that was labeled fear," even though my custom queries had no official annotations and did not contribute to the aggregate evaluation. It also described MiniLM's retrieved tweets as discussing hiring, although none explicitly mentions hiring, and used "fewer" rather than zero to describe TF-IDF's topically relevant results.

After I challenged these statements, the agent restated the correct facts, although it did not explicitly acknowledge that its earlier statements were wrong.

Overall, my experiments demonstrate that topical relevance, emotion label agreement, and retrieval diversity capture different aspects of retrieval quality. However, because my two queries concern a single topic and target emotion, these findings cannot establish the universal superiority of any retriever.

### **>>> Exercise 16 (take home):**
**Purpose:** treat an agent's explanation of model errors as a set of claims to test against evidence.

In the Master's Exercise 8 you explained your best model's most frequent confusion from the confusion matrix alone, yet never saw a misclassified tweet. 

One request per message: have the agent run a classifier (Exercise 14's best, or the Master's TF-IDF linear SVM) so there is a `result_id`, break it down with `inspect_result_tool`, then show up to five tweets from one confusion with `show_misclassified_tool` and explain why the model got them wrong. Pick the confusion from the whole matrix yourself: `most_confused` names one direction, where Exercise 8 had you add up both.

In the cell below, say which confusion you chose and why, and put the agent's explanation next to your Exercise 8 paragraph. Which of its sentences say something your tweet-blind paragraph could not have said, and which tweets support them (cite `doc_id`s). For the rest, say what in the table could have shown them wrong.

*What to check in the agent's output:* it describes the tweets in the table and not tweets it invented, and refers to them by `doc_id`.

**Summary:** The actual misclassified tweets provided only limited support for my Exercise 8 hypothesis. Non-emotional tokens and the loss of emotionally informative features appeared to contribute to errors in both directions, while lexical ambiguity and a preprocessing artifact were observed only in the fear → anger sample. Furthermore, several of the agent's linguistically plausible explanations contradicted the actual model behavior.

### 1. Confusion Selection and Experimental Design

I selected `classify_3`, the best-performing model from Exercise 14, which reproduced the Master's TF-IDF Linear SVM baseline (C = 1.0, random_state = 42, macro F1 = 0.8621).

Using `inspect_result_tool` (`inspect_13`), I confirmed that the confusion matrix matched my Master Exercise 8 results.

| Emotion Pair | A → B | B → A | Total |
|---|---:|---:|---:|
| Anger ↔ Fear | 7 | 8 | **15** |
| Anger ↔ Sadness | 7 | 3 | 10 |
| Fear ↔ Sadness | 5 | 5 | 10 |
| Fear ↔ Joy | 1 | 5 | 6 |
| Joy ↔ Sadness | 0 | 4 | 4 |
| Anger ↔ Joy | 0 | 3 | 3 |

I selected fear → anger because anger and fear were the most frequently confused pair, and fear → anger was the more frequent direction within that pair. Although this direction also matched `most_confused`, my selection was based on independently examining the complete confusion matrix.

I then used `show_misclassified_tool` to retrieve five of the eight matching errors (`misclassified_14`). To reduce confirmation bias, I deliberately withheld my Exercise 8 explanation until the agent had independently analyzed the tweets.

### 2. Comparing My Exercise 8 Explanation with the Agent's Explanation

**My original Exercise 8 paragraph:**

> Anger and fear are the most frequently confused pair, with 7 anger tweets predicted as fear and 8 fear tweets predicted as anger, giving 15 errors out of 48 total errors. The confusion is almost symmetric. A plausible linguistic explanation is that both emotions can occur in discussions of threats, conflict, or danger and therefore share emotionally intense words. Unigram TF-IDF represents weighted word occurrences without word order, so it may miss whether a person is expressing anger at a threat or fear of it. Its classifier learns associations from training labels rather than directly understanding the speaker's reaction. Shared vocabulary could therefore contribute to errors in both directions, although the confusion matrix alone cannot establish the cause and actual misclassified tweets must be inspected.

**The agent's initial explanation of `misclassified_14`:**

> These cues are unigram tokens that receive relatively high TF-IDF weights because they are not overly common across the corpus.

> Consequently, the Linear SVM learns positive coefficients for them in the anger decision function.

For document `21233`, the agent additionally proposed:

> The word “offensive” is a strong anger-associated term in many training examples.

The agent extended my original explanation by identifying specific lexical features and suggesting how they might influence classification. However, its proposed model behavior requires evidence beyond the retrieved tweets.

| My Exercise 8 claim | Agent's explanation | Relevant doc_ids | Evidence and limitation |
|---|---|---|---|
| Anger and fear may share emotional vocabulary. | Strong negative words favor anger. | 21151, 21221 | Negative expressions are present, but their classifier contributions are unknown from the tool output. |
| Unigram TF-IDF lacks contextual information. | Individual words may dominate the prediction. | 21189, 21233 | Plausible, but vocabulary inclusion and learned coefficients require verification. |
| The confusion may involve threats or conflicts. | Complaints and expressions of frustration contribute to confusion. | 21151, 21165, 21221 | Complaints are present, but explicit threats are generally absent. |

### 3. Findings That Required Actual Tweets

The following statements from the agent's explanation contain document-specific information that my tweet-blind Exercise 8 paragraph could not have identified.

| Doc ID | Fear intensity | Agent's sentence or excerpt | Textual evidence | My assessment |
|---|---:|---|---|---|
| 21151 | 0.542 | “The language conveys frustration/anger” | “are you kidding me!”, “WTF!” | Supported as a plausible emotional interpretation. |
| 21165 | 0.250 | “The wording expresses annoyance and concern about unreliable software” | “buggy”, “sporadically” | Annoyance is plausible, but concern is less explicit. |
| 21189 | 0.229 | “The act of unfollowing can signal disapproval or distancing” | “unfollowed without hesitation”, heart symbol | Plausible, but the author's actual emotion remains ambiguous. |
| 21221 | 0.521 | “The speaker expresses frustration/anger about being overcharged” | “ripping me off again”, “#awful” | Supported by the complaint's language. |
| 21233 | 0.167 | “The word ‘offensive’ is a strong anger-associated term in many training examples” | “offensive genius” | The word is present, but its training association requires separate verification. |

The agent correctly identified complaint-like language in `21151`, `21165`, and `21221`. These observations are genuinely new because my Exercise 8 paragraph could not identify the topics or specific expressions of misclassified tweets.

The agent's hypothesis concerning “offensive” in `21233` was particularly interesting. The original username, `@mikefreemanNFL`, suggests an American football context. Humans can interpret this contextual information, whereas the preprocessing procedure removes the username and unigram TF-IDF does not explicitly distinguish word senses.

The intensity values also revealed variation unavailable from the confusion matrix. Documents `21189` and `21233` had relatively low annotated fear intensities, whereas `21151` and `21221` had higher values despite containing language that I would interpret primarily as dissatisfaction.

These observations suggest that emotional ambiguity and differences between human interpretation and annotated labels deserve consideration. However, intensity measures emotional strength, not annotation confidence.

### 4. Testing and Falsifying the Agent's Claims

I evaluated the agent's explanations against the retrieved documents and the actual implementation rather than accepting them because they sounded plausible.

| Agent claim | Counterevidence or verification requirement |
|---|---|
| The tool displayed the “first five” matching errors. | `classification_tools.py` uses `errors.sample(..., random_state=SEED)`, followed by sorting the selected sample. |
| Document `21233` contains a question mark. | The original tweet has an interrogative structure but no literal question mark. |
| “terrible” is an individual feature in `21151`. | The default tokenizer does not segment `#ThisIsATerribleIdea` into individual words. |
| Strong negative words explain these misclassifications. | `21189` contains no obvious strong negative vocabulary, challenging this as a universal explanation. |
| `21189` expresses fear of further interaction. | The tweet does not mention fear of interaction or explain why the author unfollowed someone. |
| `21221` expresses financial anxiety. | The tweet establishes dissatisfaction with charges but does not establish financial anxiety. |
| The true label was anger and the prediction was fear. | Every document in `misclassified_14` has true label fear and predicted label anger. |

The emotional interpretations concerning `21189` and `21221` are not necessarily false, but their lack of textual support means they should remain speculative.

Similarly, the presence of a word does not establish that it influenced the prediction. A proposed feature-level explanation would be contradicted if the word were absent from the vocabulary or if its contribution favored the opposite emotion.

After my first follow-up challenge, the agent acknowledged that it could not independently inspect the vocabulary or coefficients. Nevertheless, it continued to associate lexical rarity with classifier weights and even reversed the classification direction for several documents.

This demonstrates that an agent's statement that a claim is verifiable does not guarantee that it has correctly verified it.

### 5. Auxiliary Check Outside the Agent

To test the model-level hypotheses, I inspected the saved vocabulary and retrained a LinearSVC using the saved TF-IDF matrix with C = 1.0 and random_state = 42. It reproduced `classify_3`'s accuracy (0.8617), the count of eight fear → anger errors, and the misclassifications of all five sampled documents.

I calculated each feature's contribution to the difference between the anger and fear decision scores:

**Contribution(j) = x(j) × [w(anger,j) − w(fear,j)]**

Here, x(j) is the document's TF-IDF feature value, while w represents the learned classifier coefficients. A positive contribution favors anger relative to fear, whereas a negative contribution favors fear relative to anger.

| Doc ID | Major contributions favoring anger | Evidence against the agent's explanation |
|---|---|---|
| 21151 | black (+0.376), wtf (+0.231), rock (+0.204) | “terrible” is not an independent token; kidding contributes −0.167. |
| 21165 | tv (+0.304), app (+0.232) | buggy and sporadically are out of vocabulary. |
| 21189 | lt (+0.299), without (+0.161) | unfollowed and hesitation are out of vocabulary. |
| 21221 | 30 (+0.315), flight (+0.195) | awful contributes −0.689; ripping also favors fear. |
| 21233 | offensive (+0.413), sort (+0.371) | The offensive hypothesis is supported, but genius is out of vocabulary. |

These results revealed several factors that the agent's initial explanation overlooked:

- **Lexical ambiguity:** “black” and “rock” in `21151` are parts of the names Jack Black and The Rock.
- **Preprocessing artifacts:** The HTML entity `&lt;3` in `21189` produces the token “lt,” which contributes toward anger.
- **Non-emotional features:** Numeric strings and ordinary words, such as “30,” “flight,” and “app,” contribute to the decision.
- **Human–model disagreement:** Words such as “awful” and “kidding,” which humans might associate with anger, actually favor fear relative to anger in these documents.

The results support some of the agent's hypotheses, particularly those involving “wtf” and “offensive,” but contradict many others. Moreover, “offensive” appeared in only two training tweets, both labeled anger, making the agent's claim about “many training examples” misleading.

This illustrates **explanation faithfulness**: a linguistically plausible explanation is not necessarily faithful to the classifier's actual decision process. However, individual feature contributions should not be treated as complete causal explanations of a multiclass prediction.

### 6. Reverse-Direction Experiment: Anger → Fear

To investigate my original bidirectional hypothesis, I retrieved five of the seven anger → fear errors (`misclassified_15`).

The following feature contributions were also calculated outside the agent, with positive values now favoring fear relative to anger.

| Doc ID | Textual observation | Major contributions favoring fear |
|---|---|---|
| 10861 | Anger expressed through threats involving someone's job and imprisonment. | job (+0.160), something (+0.140) |
| 10867 | Complaint about game lag. | please (+0.264), trash (+0.253) |
| 10879 | Possible frustration about not receiving an invitation. | was (+0.341) |
| 10905 | The author accidentally scared a cat. | times (+0.289), trying (+0.281), scared (+0.251) |
| 10907 | Angry emojis expressing dissatisfaction. | was (+0.234), sorted (+0.233) |

Only `10861` and `10905` provided relatively concrete partial support for my Exercise 8 hypothesis. In `10861`, “job” contributes toward fear within a threatening context, although “federal” and “prison” are out of vocabulary.

In `10905`, “scared” contributes toward fear, but the experiencer of fear is the cat rather than the author. This illustrates the absence of semantic-role information in unigram TF-IDF. Nevertheless, the non-emotional words “times” and “trying” contribute more strongly.

Document `10907` instead demonstrates information loss: the tokenizer discards angry emojis, lowercase conversion removes capitalization-based emphasis, and “hufflepuff” is out of vocabulary.

I also identified additional errors in the agent's analysis of `misclassified_15`:

- It suggested that the username in `10879` might influence classification, although `clean_text()` removes usernames.
- It claimed that the tokenizer retains the `#` symbol, contradicting the default token pattern.
- It listed “unreliable Android app” as lexical evidence for `21165`, although “unreliable” came from my summary rather than the original tweet.

The agent's response also ended prematurely at “Vocabulary composition.”

These observations reinforce the importance of distinguishing original textual evidence from summaries, hypothetical examples, and unsupported model explanations.

Across both directions, the sampled errors exhibit similar feature-level limitations, particularly substantial contributions from non-emotional tokens and the loss of emotionally informative features. Nearly symmetric confusion counts therefore do not establish that shared threat vocabulary is the underlying mechanism.

### 7. Conclusion and Limitations

My Exercise 8 hypothesis was plausible but too narrow. Among the ten examined tweets, only two provided relatively concrete partial support for the shared-vocabulary explanation. Other cases revealed lexical ambiguity, preprocessing artifacts, vocabulary limitations, and discrepancies between human emotional interpretation and learned feature associations.

I distinguished three levels of evidence throughout this investigation:

- **Confusion matrix:** identifies where classification errors occur.
- **Textual analysis:** provides possible linguistic explanations.
- **Feature contribution analysis:** tests whether those explanations reflect the classifier's decision scores.

This investigation was limited to reproducibly sampled subsets of five out of eight fear → anger errors and five out of seven anger → fear errors. The auxiliary feature analysis was performed outside the agent. Furthermore, emotion intensity is not annotation confidence, and neither these samples nor their intensity values establish general conclusions about all classification errors.

Ultimately, I learned that evaluating an agent's explanation requires more than checking whether it sounds reasonable. Its claims must be tested against the original documents, the actual preprocessing implementation, and, when available, the classifier's learned features.

## 5. Submitting your session log
**Objective:** confirm the log exists before you submit it.

Your conversation is logged turn by turn to `session_logs/lab2_agent_pipeline_session.jsonl.enc`, encrypted with the course's grading public key as it is written. You cannot read or edit it; upload it exactly as produced, together with this notebook (Exercises 14 to 16 answered) and the Master notebook's log, `session_logs/lab2_main_session.jsonl.enc`. `session_logs/` is git-ignored, so neither log will be in your repository: both are handed in on NTU COOL, inside the ZIP.

For the complete lab submission, push your notebooks to your repository and submit its link plus the ZIP on NTU COOL by **November 25 at 11:59 pm**. If your ZIP turns out to be missing a log, you can upload it until **November 26 at 11:59 pm**. See `README.md` for the ZIP contents.

In [4]:
import os

log_path = "session_logs/lab2_agent_pipeline_session.jsonl.enc"
if os.path.exists(log_path):
    print(f"Log file present: {log_path} ({os.path.getsize(log_path)} bytes)")
else:
    print("No log yet. Send at least one message in the chat above first.")

Log file present: session_logs/lab2_agent_pipeline_session.jsonl.enc (210428 bytes)
